Here we test some evaluation metrics on the different files we have in our predictions folder, so comparing how the model performance changes depending on the type of test set and the model we used. 

Intial approach: Use the given span_f1 for all files 

Taha's brain: A metric depending on the specific tag we changed so for ex. If we changed the names to names from different regions, then how evaluating on just B-PER and I-PER predictions, whereas if we changed the locations then evaluating only on the B-LOC I-LOC tags, then we have things like random strings and typos there we can just compare with a simple f1 score.

An example metric here could be f1 on that specific tag

The upside of the initial approach is it is universally comparable, easy to code (code is already there just need to make adjustments because our gold and predictions are in the same file (easy peezy))

The second approach might be better but is probably a bit harder to write the code for.

Below I will do the span f1 on the original test set

In [148]:
def readNlu(path):
    """Reads given iob2 file based on path, returns gold truth and predictions as seperate lists
    ----------
    path : str
        path to an iob2 file where the second column is the ground truth and the third column is the predictions
    
    Returns
    ----------
    annotations : list
        a list with all the ground truths where each list within is a seperate sentence
    
    predicition : list
        a list with all the predictions where each list within is a seperate sentence
    """
    annotations = []
    cur_annotation = []

    prediction = []
    cur_prediction = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()
        if line == '':
            annotations.append(cur_annotation)
            cur_annotation = []

            prediction.append(cur_prediction)
            cur_prediction = []
        elif line[0] == '#' and len(line.split(' ')) == 1:
            continue
        else:
            cur_annotation.append(line.split(' ')[1])
            cur_prediction.append(line.split(' ')[2])
    return annotations, prediction

In [149]:
an, pr = readNlu("../predictions/original_test/mono/test_conll_results_mono.iob2")

In [150]:
def read_iob2_file(path):
    """
    Read provided Universal NER iob2 file
    
    :param path: path to read from
    :returns: list with sequences of words and NER labels for each sentence
    """
    data = []
    gold_ner_tags = []
    predicted_ner_tags = []

    for line in open(path, encoding='utf-8'):
        line = line.strip()

        if line:
            if line[0] == '#':
                continue # skip comments
            tok = line.split(' ')
            #print(tok)
            gold_ner_tags.append(tok[1])
            predicted_ner_tags.append(tok[2])
        else:
            if gold_ner_tags:  # skip empty lines
                data.append((gold_ner_tags, predicted_ner_tags))
            gold_ner_tags = []
            predicted_ner_tags = []

    # check for last one
    if gold_ner_tags != []:
        data.append((gold_ner_tags, predicted_ner_tags))
    return data

d = read_iob2_file("../predictions/original_test/mono/test_conll_results_mono.iob2")

In [151]:
len(an) == len(pr) == len(d)

True

FUNCTION CHANGED SUCCESFULLY, Time to test the whole thing

In [152]:
#all the other functions are the same

def toSpans(tags):
    # Converts a list of tags to a list of spans
    # in: ['B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-ORG', 'I-ORG', 'O']
    # out: {'7-9:ORG', '0-2:PER'}
    spans = set()
    for beg in range(len(tags)):
        if tags[beg][0] == 'B':
            end = beg
            for end in range(beg+1, len(tags)):
                if tags[end][0] != 'I':
                    break
            spans.add(str(beg) + '-' + str(end) + ':' + tags[beg][2:])
    return spans

def getBegEnd(span):
    return [int(x) for x in span.split(':')[0].split('-')]

def getLooseOverlap(spans1, spans2):
    # returns the overlap of spans without taking the exact boundaries
    # into account. If entities overlap they also count as found.
    found = 0
    for spanIdx, span in enumerate(spans1):
        spanBeg, spanEnd = getBegEnd(span)
        label = span.split(':')[1]
        match = False
        for span2idx, span2 in enumerate(spans2):
            span2Beg, span2End = getBegEnd(span2)
            label2 = span2.split(':')[1]
            if label == label2:
                if span2Beg >= spanBeg and span2Beg <= spanEnd:
                    match = True
                if span2End <= spanEnd and span2End >= spanBeg:
                    match = True
        if match:
            found += 1
    return found

def getUnlabeled(spans1, spans2):
    # Counts the overlap in spans after removing the labels
    return len(set([x.split(':')[0] for x in spans1]).intersection([x.split(':')[0] for x in spans2]))

In [165]:
s = toSpans(an[10])
print(s)
s = {i for i in s if i.split(":")[1] != "MISC"}
print(s)

{'0-2:PER', '14-16:PER', '19-20:MISC', '23-25:PER'}
{'0-2:PER', '14-16:PER', '23-25:PER'}


In [166]:
#Essentially what's at the end of span_f1.py, a strict scoring system where all tags should match, 
#a loose system where even if one of the bios tag is seen so if IT Univerisity of Copenhagen and only Univerisity is detceted
#then there is still credit given, lastly the unlabelled scoring where if an entity is matched regardless of what it is,
#credit is then given, this will be especially useful in the random strings test set

def evaluate_specific(file_path,ner_tag_measured):
    """Takes a file_path and the ner_tag that was modified in that specific set, then returns the metrics which only look 
    at the models performance on that specific tag
        ----------
    file_path : str
        path to the file you want to measure the performance over 
    
    ner_tag_measured : str
        The group which that specific file was measuring without the B or I so just "PER" or "MISC"

    Returns
    -------
    metrics: int
        a strict precison, recall, f1 score (first 3 numbers), if ner_tag specified then only for those tags
        a loose precison, recall, f1 score (second 3 numbers), if ner_tag specified then only for those tags
        an unlabelled precison, recall, f1 score (last 3 numbers), over the entire dataset regardless of ner_tag
    """
    gold_ners, pred_ners = readNlu(file_path)

    tp = 0
    fp = 0
    fn = 0

    recall_loose_tp = 0
    recall_loose_fn = 0
    precision_loose_tp = 0
    precision_loose_fp = 0

    tp_ul = 0
    fp_ul = 0
    fn_ul = 0 

    for gold_ner, pred_ner in zip(gold_ners, pred_ners):
        gold_spans = toSpans(gold_ner)
        pred_spans = toSpans(pred_ner)

        overlap_ul = getUnlabeled(gold_spans, pred_spans)
        tp_ul += overlap_ul
        fp_ul += len(pred_spans) - overlap_ul
        fn_ul += len(gold_spans) - overlap_ul
        
        if ner_tag_measured:
            gold_spans = {i for i in gold_spans if i.split(":")[1] != ner_tag_measured}
            pred_spans = {i for i in pred_spans if i.split(":")[1] != ner_tag_measured}

        overlap = len(gold_spans.intersection(pred_spans))
        tp += overlap
        fp += len(pred_spans) - overlap
        fn += len(gold_spans) - overlap

        overlap_loose = getLooseOverlap(gold_spans, pred_spans)
        recall_loose_tp += overlap_loose
        recall_loose_fn += len(gold_spans) - overlap_loose

        overlap_loose = getLooseOverlap(pred_spans, gold_spans)
        precision_loose_tp += overlap_loose
        precision_loose_fp += len(pred_spans) - overlap_loose


    prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    l_prec = 0.0 if precision_loose_tp + precision_loose_fp == 0 else precision_loose_tp/(precision_loose_tp+precision_loose_fp)
    l_rec = 0.0 if recall_loose_tp+recall_loose_fn == 0 else recall_loose_tp/(recall_loose_tp+recall_loose_fn)
    l_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    tp = tp_ul
    fp = fp_ul
    fn = fn_ul

    ul_prec = 0.0 if tp+fp == 0 else tp/(tp+fp)
    ul_rec = 0.0 if tp+fn == 0 else tp/(tp+fn)
    ul_f1 = 0.0 if prec+rec == 0.0 else 2 * (prec * rec) / (prec + rec)

    return prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1

        

In [44]:
import os 

directory = "../predictions"
folders = os.listdir(directory)

meta_data = []
for folder in folders:
    models = os.listdir(os.path.join(directory,folder))
    for model in models:
        files = os.listdir(os.path.join(directory,folder,model))
        for file in files:
            path = os.path.join(directory,folder,model,file)
            sub_category = file.split("_")[0]

            meta_data.append({
                "folder" : folder,
                "model" : model,
                "file" : file,
                "path" : path,
                "sub_categories" : sub_category
            })

df = pd.DataFrame(meta_data)

cat_2_ner = {'gender_names': 'PER',
 'location_exonym_endonym': 'LOC',
 'person': 'PER',
 'pronouns': False,
 'random': False,
 'original_test': False,
 'typos': False,
 'typos_entity' : 'PER'}

df['modified_ner'] = df["folder"].map(cat_2_ner)

df.to_csv("../file_metadata.csv")



In [45]:
from span_f1_adjusted import evaluate_specific

metadata = pd.read_csv("../file_metadata.csv")

results_data = []
for i in range(len(metadata)):
    file_path, modifed_ner = metadata["path"].iloc[i], metadata["modified_ner"].iloc[i]
    folder, model, file = metadata["folder"].iloc[i], metadata["model"].iloc[i], metadata["file"].iloc[i]

    prec, rec, f1, l_rec, l_prec, l_f1, ul_rec, ul_prec, ul_f1 = evaluate_specific(file_path,modifed_ner)

    results_data.append({
        "folder" : folder,
        "model" : model,
        "modified_ner" : modifed_ner,
        "file" : file,
        "path" : file_path,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "loose_precision": l_prec,
        "loose_recall": l_rec,
        "loose_f1": l_f1,
        "unlabeled_precision": ul_prec,
        "unlabeled_recall": ul_rec,
        "unlabeled_f1": ul_f1
        })

results = pd.DataFrame(results_data)

results.to_csv("../evaluation_results.csv")

Someone else can analyze the results Peace out

In [127]:
original_mask = ((df["sub_categories"] == "test"))
base_results = df[original_mask].groupby("file").mean(numeric_only=True)
base_results

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
test_conll_results_mono.iob2,0.673139,0.662890,0.667975,0.869831,0.860482,0.667975,0.710356,0.699540,0.667975
test_conll_results_multi.iob2,0.665510,0.577373,0.618316,0.885714,0.764518,0.618316,0.699796,0.607118,0.618316


In [108]:
gender_mono_mask = ((df["folder"] == "gender_names") & (df["model"] == "mono"))
gender_mono_results = df[gender_mono_mask].groupby(df["sub_categories"]).mean(numeric_only=True)
gender_mono_results

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,
female,0.685690,0.675637,0.680626,0.867624,0.859348,0.680626,0.724970,0.714341,0.680626
male,0.687272,0.677355,0.682277,0.870206,0.862199,0.682277,0.724243,0.713792,0.682277


In [109]:
gender_multi_mask = ((df["folder"] == "gender_names") & (df["model"] == "multi"))
gender_multi_results = df[gender_multi_mask].groupby(df["sub_categories"]).mean(numeric_only=True)
gender_multi_results

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,
female,0.675708,0.587128,0.628311,0.879576,0.76119,0.628311,0.715034,0.62130,0.628311
male,0.678560,0.590652,0.631561,0.884670,0.76574,0.631561,0.713586,0.62114,0.631561


In [136]:
pronouns_mono_mask = ((df["folder"] == "pronouns") & (df["model"] == "mono"))
pronouns_mono_results = df[pronouns_mono_mask].groupby("file").mean(numeric_only=True)
pronouns_mono_results

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
female_pronouns_test_results_mono.iob2,0.673260,0.662890,0.668035,0.869808,0.860305,0.668035,0.710484,0.699540,0.668035
male_pronouns_test_results_mono.iob2,0.673198,0.663067,0.668094,0.869854,0.860659,0.668094,0.710228,0.699540,0.668094
neutral_pronouns_test_results_mono.iob2,0.672842,0.662358,0.667559,0.869424,0.859773,0.667559,0.710432,0.699363,0.667559


In [135]:
pronouns_multi_mask = ((df["folder"] == "pronouns") & (df["model"] == "multi"))
pronouns_multi_results = df[pronouns_multi_mask].groupby("file").mean(numeric_only=True)
pronouns_multi_results

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
female_pronouns_test_results_multi.iob2,0.665850,0.577550,0.618565,0.886099,0.764873,0.618565,0.699939,0.607118,0.618565
male_pronouns_test_results_multi.iob2,0.665714,0.577550,0.618506,0.885918,0.764873,0.618506,0.699796,0.607118,0.618506
neutral_pronouns_test_results_multi.iob2,0.665238,0.577018,0.617996,0.885487,0.764164,0.617996,0.699939,0.607118,0.617996


In [ ]:
location_mono_mask = ((df["folder"] == "location") & (df["model"] == "mono"))
location_mono_result = df[location_mono_mask].groupby("file").mean(numeric_only=True)
location_mono_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
location_endonym_results_mono.iob2,0.640886,0.625000,0.632843,0.836964,0.820644,0.632843,0.698439,0.681126,0.632843
location_latin_results_mono.iob2,0.635084,0.623052,0.629011,0.823137,0.812500,0.629011,0.702942,0.689625,0.629011


In [121]:
location_multi_mask = ((df["folder"] == "location") & (df["model"] == "multi"))
location_multi_result = df[location_multi_mask].groupby("file").mean(numeric_only=True)
location_multi_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
location_endonym_results_multi.iob2,0.639754,0.552762,0.593085,0.869672,0.748052,0.593085,0.684426,0.591360,0.593085
location_latin_results_multi.iob2,0.639606,0.552408,0.592818,0.852194,0.732826,0.592818,0.698237,0.603045,0.592818


In [133]:
random_mono_mask = ((df["folder"] == "random") & (df["model"] == "mono"))
random_mono_result = df[random_mono_mask].groupby("file").mean(numeric_only=True)
random_mono_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
random_conll_results_mono.iob2,0.36543,0.190156,0.250146,0.507315,0.262571,0.250146,0.655325,0.341006,0.250146


In [134]:
random_multi_mask = ((df["folder"] == "random") & (df["model"] == "multi"))
random_multi_result = df[random_multi_mask].groupby("file").mean(numeric_only=True)
random_multi_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
file,,,,,,,,,
random_conll_results_multi.iob2,0.455624,0.169972,0.247582,0.633602,0.237429,0.247582,0.673944,0.251416,0.247582


In [131]:
person_mono_mask = ((df["folder"] == "person") & (df["model"] == "mono"))
person_mono_result = df[person_mono_mask].groupby("sub_categories").mean(numeric_only=True)
person_mono_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,
african,0.672799,0.662890,0.667807,0.860227,0.851735,0.667807,0.716844,0.706285,0.667807
american,0.686400,0.677222,0.681780,0.868195,0.861244,0.681780,0.726499,0.716785,0.681780
arabic,0.689053,0.679586,0.684286,0.871930,0.864536,0.684286,0.726591,0.716608,0.684286
european,0.681187,0.670042,0.675569,0.869483,0.860535,0.675569,0.717313,0.705577,0.675569
indian,0.689795,0.680329,0.685029,0.869922,0.862358,0.685029,0.729288,0.719281,0.685029


In [132]:
person_multi_mask = ((df["folder"] == "person") & (df["model"] == "multi"))
person_multi_result = df[person_multi_mask].groupby("sub_categories").mean(numeric_only=True)
person_multi_result

,precision,recall,f1,loose_precision,loose_recall,loose_f1,unlabeled_precision,unlabeled_recall,unlabeled_f1
sub_categories,,,,,,,,,
african,0.668678,0.576735,0.619312,0.874409,0.755205,0.619312,0.711192,0.613403,0.619312
american,0.680142,0.590285,0.632035,0.881750,0.763964,0.632035,0.719178,0.624163,0.632035
arabic,0.684676,0.592351,0.635176,0.885048,0.767440,0.635176,0.721942,0.624593,0.635176
european,0.674745,0.586579,0.627581,0.887393,0.767812,0.627581,0.706212,0.613934,0.627581
indian,0.687131,0.593892,0.637118,0.884116,0.765687,0.637118,0.724844,0.626487,0.637118


In [140]:
df.to_csv("../ner_evaluation_results.csv")